In [ ]:
import os

print(os.getcwd())
print(os.listdir())


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/workspaces/LLM-zoomcamp-homework/02/llm-zoomcamp-hw5/.env"),
override=True


print(os.getenv("OPENAI_API_KEY"))

In [ ]:
import os

print(os.getenv("OPENAI_API_KEY"))

In [ ]:
import sys
print(sys.executable)

In [ ]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [ ]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

In [ ]:

from rag_helper import RAGBase
class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            return response

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from starter import index, client

load_dotenv("/workspaces/LLM-zoomcamp-homework/02/llm-zoomcamp-hw5/.env")

client = OpenAI()

rag = RAGTraced(
    index=index,
    llm_client=client
)

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

print(answer)

In [ ]:
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult, SimpleSpanProcessor
from opentelemetry import trace


class ListSpanExporter(SpanExporter):
    def __init__(self):
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass


list_exporter = ListSpanExporter()

trace.get_tracer_provider().add_span_processor(
    SimpleSpanProcessor(list_exporter)
)

In [ ]:
list_exporter.spans.clear()

query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

span_names = [span.name for span in list_exporter.spans]

print(span_names)
print("Broj spanova:", len(span_names))

In [ ]:
rag = RAGTraced(
    index=index,
    llm_client=client
)

In [ ]:
list_exporter.spans.clear()

query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

llm_spans = [span for span in list_exporter.spans if span.name == "llm"]

for span in llm_spans:
    print("Span name:", span.name)
    print("Input tokens:", span.attributes.get("input_tokens"))
    print("Output tokens:", span.attributes.get("output_tokens"))

In [ ]:
list_exporter.spans.clear()

query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

for span in list_exporter.spans:
    duration_ms = (span.end_time - span.start_time) / 1_000_000
    print(span.name, round(duration_ms, 2), "ms")

In [ ]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [ ]:
provider = trace.get_tracer_provider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [ ]:
answer = rag.rag(query)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

pd.read_sql_query(
    "select distinct name from spans",
    conn,
)

In [ ]:
list_exporter.spans.clear()
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

for span in list_exporter.spans:
    duration_ms = (span.end_time - span.start_time) / 1_000_000
    print(span.name, round(duration_ms, 2), "ms")

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

pd.read_sql_query("""
SELECT
    name,
    SUM(end_time_unix_nano - start_time_unix_nano)/1000000.0 as total_ms
FROM spans
WHERE name != 'rag'
GROUP BY name
ORDER BY total_ms DESC
""", conn)



In [ ]:
pd.read_sql_query("""
SELECT
    name,
    SUM(end_time - start_time) / 1000000.0 AS total_ms
FROM spans

GROUP BY name
ORDER BY total_ms DESC
""", conn)

In [ ]:
for _ in range(3):
    rag.rag(query)

In [ ]:
pd.read_sql_query("""
SELECT input_tokens
FROM spans
WHERE name = 'llm'
""", conn)

In [ ]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

In [ ]:
from agent import agent

result = agent.run_sync("How do I run Ollama locally?")

print(result.output)

In [ ]:
from dotenv import load_dotenv
import os
import logfire

load_dotenv()

print(os.getenv("LOGFIRE_TOKEN"))

In [ ]:
from dotenv import dotenv_values

dotenv_values("/workspaces/LLM-zoomcamp-homework/02/llm-zoomcamp-hw5/.env")

In [ ]:
import os

for key in ["OPENAI_API_KEY", "LOGFIRE_TOKEN", "LOGFIRE_READ_TOKEN"]:
    os.environ.pop(key, None)

In [ ]:
from dotenv import load_dotenv
import os

env_path = "/workspaces/LLM-zoomcamp-homework/02/llm-zoomcamp-hw5/.env"

loaded = load_dotenv(env_path, override=True)

print("Loaded:", loaded)
print("OPENAI:", repr(os.getenv("OPENAI_API_KEY")))
print("LOGFIRE:", repr(os.getenv("LOGFIRE_TOKEN")))
print("LOGFIRE READ:", repr(os.getenv("LOGFIRE_READ_TOKEN")))

In [ ]:
import logfire
import os

logfire.configure(
    token=os.getenv("LOGFIRE_TOKEN")
)

logfire.instrument_pydantic_ai()

print("Logfire configured")

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

print(len(documents))
print(type(index))

In [ ]:
result = await faq_agent.run(
    "How do I run Ollama locally?",
    deps=SearchDeps(index=index)
)

print(result.output)


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("READ TOKEN:", os.getenv("LOGFIRE_READ_TOKEN") is not None)

In [ ]:
import duckdb
import pandas as pd

db_path = "logfire_pipeline.duckdb"

conn = duckdb.connect(db_path)

tables_df = conn.sql("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'agent_traces'
ORDER BY table_name;
""").df()

display(tables_df)

print("Broj tabela:", len(tables_df))

In [ ]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

df = conn.sql("""
DESCRIBE agent_traces.traces
""").df()

df

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("logfire_pipeline.duckdb")

df_desc = conn.sql("""
DESCRIBE agent_traces.traces
""").df()

display(df_desc)

In [ ]:
cols = df_desc["column_name"].tolist()

matches = [
    c for c in cols
    if "token" in c.lower()
    or "usage" in c.lower()
    or "gen_ai" in c.lower()
    or "attribute" in c.lower()
    or "trace" in c.lower()
    or "name" in c.lower()
]

matches

In [ ]:
for col in df.columns:
    sample = df[col].astype(str)

    if sample.str.contains("gen_ai.usage.input_tokens", na=False).any():
        print("Found in column:", col)

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("logfire_pipeline.duckdb")

df = conn.sql("""
SELECT *
FROM agent_traces.traces
""").df()

print("Rows:", len(df))
print("Columns:", len(df.columns))

for col in df.columns:
    print(col)

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("logfire_pipeline.duckdb")

df = conn.sql("""
SELECT *
FROM agent_traces.traces
LIMIT 5
""").df()

display(df)

print("Rows:", len(df))
print("Columns:", len(df.columns))

for col in df.columns:
    print(col)

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("logfire_pipeline.duckdb")

df = conn.sql("""
SELECT *
FROM agent_traces.traces
LIMIT 5
""").df()

display(df)

print("Rows:", len(df))
print("Columns:", len(df.columns))

for col in df.columns:
    print(col)

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("logfire_pipeline.duckdb")

df = conn.sql("""
SELECT *
FROM agent_traces.traces
""").df()

token_cols = [
    col for col in df.columns
    if "token" in col.lower()
    or "usage" in col.lower()
    or "gen_ai" in col.lower()
]

print("Token/usage columns:")
for col in token_cols:
    print(col)